In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly

## PVL project information

we clean up *pvl* patient with DTI and bayley here.

In [2]:
bayley = pd.read_excel('../../Annotations/patient_information/Bayley III follow-up_2018.9.5.xlsx')

In [3]:
bayley.columns = bayley.columns[:2].tolist() + bayley.iloc[0, 2:].tolist()
bayley = bayley.iloc[1:, :]
#bayley.iloc[:, 0] = bayley.iloc[:, 0].map(lambda x:str(int(x)))
bayley.iloc[:, 1] = bayley.iloc[:, 1].map(lambda x:str(int(x)))

In [4]:
new_sequence = pd.read_excel('../../Annotations/patient_information/Copy of all new seqs_2017.9.19.xlsx')

In [5]:
new_sequence.loc[:, '3T_id'] = new_sequence.loc[:, '3T_id'].map(lambda x: str(int(x)))
print(new_sequence.shape)

(396, 69)


In [6]:
final = bayley.merge(new_sequence, left_on=['影像号'], right_on=['3T_id'], how='outer')

In [7]:
select = final.loc[:, 'MRI特征'].str.contains('软化')
select[pd.isnull(select)] = False

select2 = final.loc[:, 'MRI诊断'].str.contains('软化')
select2[pd.isnull(select2)] = False

In [8]:
pvl = final.loc[select, :]

In [9]:
final.shape

(398, 81)

some of the MRI diagnosis is missing, search in a overall list.

In [11]:
all_images_before2017 = pd.read_excel("/data1/qinqian/MRI/Annotations/patient_information/新生儿颅脑MR.xls")

In [12]:
image2diagnosis = dict()

for i in range(all_images_before2017.shape[0]):
    image2diagnosis[all_images_before2017.iloc[i, 1]] = all_images_before2017.iloc[i, -1]

In [13]:
#final.loc[:, '影像号']
#final.loc[:, '住院号']

final_diagnosis = final.loc[:, '影像号'].map(lambda x: image2diagnosis[int(x)] if (not pd.isnull(x)) and int(x) in image2diagnosis else "")

In [14]:
#final_diagnosis.shape, final.shape
final_diagnosis[final_diagnosis.str.contains('软化')]

49                        右额叶出血软化中，双侧脑室室管膜下出血，双侧脑室饱满。随访。
81                            左侧颞叶卵圆形囊性灶，血管间隙增宽？小软化灶？随访。
130                             双侧脑室旁多发软化灶，双顶部硬膜下积脓？随访。 
150    双侧基底节区少许点状脑脊液相仿信号，扩大血管间隙？，小软化灶？，必要时随访。\n双侧乳突内少...
157                           脑损伤随访：脑内多发软化灶较前明显；双侧侧脑室扩张。
158                           脑损伤随访：脑内多发软化灶较前明显；双侧侧脑室扩张。
184                     双额叶软化灶，右顶叶损伤，右侧脑室室管膜下出血，双侧脑室略扩张。
234                早产儿脑改变，双侧侧脑室旁出血灶伴部分软化，双侧脑室室管膜下出血；随访。 
245                多囊性脑软化灶，双基底节区、丘脑、内囊后肢损伤，脑室扩张，提示硬膜下积脓。
Name: 影像号, dtype: object

In [15]:
select = final.loc[:, 'MRI诊断'].str.contains('软化')
select[pd.isnull(select)] = False

pvl_recheck = final.loc[
    final_diagnosis.str.contains('软化') | select, :]

pvl_recheck = pvl_recheck.loc[~pd.isnull(pvl_recheck.iloc[:, 1]), :]



In [18]:
newsequence_with_quality = pd.read_csv('/data1/qinqian/MRI/hyperbilirubinemia/scripts/patient_info.csv')

In [19]:
pvlall.iloc[:, 1].map(int).isin(
    newsequence_with_quality.loc[:, 'patient_id'][newsequence_with_quality.loc[:, 'DTI_quality']==1])

37     False
49     False
81      True
113     True
130    False
131    False
140    False
150    False
157    False
158    False
165    False
184    False
188    False
229    False
234     True
245    False
Name: 影像号, dtype: bool

In [21]:
pvlall.iloc[:, 1].shape
pvlall.to_csv('pvl_dti_with_bayleyIII.xls')